# M1a-FULL — full-scale MSD base + pull-push term (Colab, resume-safe)

Runs **M1a-FULL**: the original MSD recipe from the upstream appendix, plus the *same*
pull-push term M1a used. Everything writes **straight to Drive**, and the run is
**resume-safe**: if the Colab runtime drops, just re-run this notebook top-to-bottom — it
picks up from the last completed epoch. **You never lose more than one epoch of work.**

**Recipe (hard-locked, from `locuslab/robust_union` `CIFAR10/train.py` + `cifar_funcs.py`)**

| | value | source |
|---|---|---|
| epochs | **50** | their `train.py` |
| batch / wd / momentum | **128 / 5e-4 / 0.9** | their `train.py` |
| lr | **one-cycle, peak 0.1** — `np.interp(t, [0,20,40,50], [0,0.1,0.005,0])` | their `train.py` |
| MSD attack | **`msd_v0`, 50 iters**, alphas (0.003, 0.05, 0.05) | their `cifar_funcs.py` |
| threat triple | **ℓ∞ 8/255 · ℓ2 0.5 · ℓ1 12** | **ours** (deliberate deviation — see below) |
| term (== M1a) | 3 APGD views @ 10 steps, α=β=0.5, τ=0.1, warm-up 10 ep, head 512→512→128 | imported verbatim from the M1a trainer |
| seed | **0** | |

> **One threat triple everywhere: ℓ∞ 8/255, ℓ2 0.5, ℓ1 12.** Upstream trains at ℓ∞
> **0.03**; we deliberately do not. Everything else in the base recipe is theirs — only the
> radius changes. Training, the val-select proxy, and the audit therefore all use the *same*
> triple, so M1a-FULL is directly comparable to M1a / M0 (which also trained at 8/255) and
> there is no train/audit threat-model gap to caveat. The triple is **imported** from the
> M1a trainer rather than restated, so it cannot silently drift from the other arms.

> **Why not `torch.OneCycleLR`:** their schedule is a plain `np.interp` of the epoch, so it
> is a pure function of the epoch counter — nothing to checkpoint, and resume restores it
> exactly. That is a feature, not a shortcut.

## Cell 1 — Setup: GPU · Drive · code

**Pick a GPU first:** Runtime → Change runtime type → GPU. **Prefer A100** if Colab offers
it; the MSD attack at 50 iters is ~5× the cost of the 10-iter version, so the GPU tier
dominates the wall-clock (see the ETA the training cell prints).

Code comes from **`attackdro_code.zip` on Drive**, the same pattern the audit notebooks
use — the GitHub repo is private, so Colab cannot clone it anonymously, and making it
public during anonymous review would be a bad trade. Re-upload the zip whenever the
trainer changes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv 2>/dev/null || echo 'NO GPU -- Runtime > Change runtime type > GPU'

from google.colab import drive
drive.mount('/content/drive')

import zipfile, os
DRIVE    = '/content/drive/MyDrive/attackdro'
CODE_ZIP = f'{DRIVE}/attackdro_code.zip'
REPO     = '/content/attackdro'

assert os.path.exists(CODE_ZIP), (
    f'{CODE_ZIP} not found on Drive. Upload the rebuilt attackdro_code.zip '
    '(it must contain scripts/dev/train_full_msd.py).')
os.makedirs(REPO, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall(REPO)
    names = z.namelist()

need = ['scripts/dev/train_full_msd.py', 'scripts/dev/c5_fromscratch.py',
        'src/robustdro/attacks/norms.py', 'src/robustdro/attacks/apgd_train.py',
        'src/robustdro/models/preact_resnet.py']
missing = [f for f in need if f not in names]
assert not missing, (f'code zip is STALE -- missing {missing}. '
                     'Rebuild and re-upload attackdro_code.zip.')

import torch
print('code    :', REPO, f'({len(names)} files from the zip)')
print('torch   :', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Cell 1b — W&B login (track the run live)

Logs to project **`attackdro-union`**, run **`C5_M1a_full_seed0`**. The run id is
deterministic and the logger uses `resume="allow"`, so every Colab re-attach **continues the
same run** — one continuous curve, not a new run per session.

**Set the Colab secret once** (sidebar 🔑 → add `WANDB_API_KEY`, value from
<https://wandb.ai/authorize>). Then every future re-attach logs in automatically, with no
prompt — which matters, because a ~13 h run will re-attach several times. Without W&B the
training still runs and still prints to stdout; it just isn't tracked.

In [ ]:
import subprocess, importlib.util
if importlib.util.find_spec('wandb') is None:
    subprocess.run(['pip', 'install', '-q', 'wandb'])
import wandb

key = None
try:
    from google.colab import userdata
    key = userdata.get('WANDB_API_KEY')          # sidebar key icon -> add this secret
except Exception:
    pass

if key:
    wandb.login(key=key)
    print('wandb: logged in from the Colab secret -- re-attaches will be automatic.')
else:
    print('No WANDB_API_KEY Colab secret found; falling back to an interactive login.')
    print('Strongly recommended: sidebar key icon -> add secret WANDB_API_KEY')
    print('(value from https://wandb.ai/authorize). Otherwise you must paste the key')
    print('again after EVERY Colab disconnect, and this run will see several.')
    wandb.login()

print('\nproject: attackdro-union | run: C5_M1a_full_seed0 (resumes in place)')

## Cell 2 — Config (Drive-direct: every write lands on Drive)

In [ ]:
DRIVE  = '/content/drive/MyDrive/attackdro'
OUTDIR = f'{DRIVE}/C5_full/M1a_full'          # ckpt_latest.pt, val_best.pt, train.json all land HERE
CIFAR_TARGZ = f'{DRIVE}/cifar-10-python.tar.gz'
SEED = 0
import os; os.makedirs(OUTDIR, exist_ok=True)
print('OUTDIR (Drive-direct):', OUTDIR)
print('  -> ckpt_latest.pt  written after EVERY epoch (resume point)')
print('  -> val_best.pt     best val worst-union so far')
print('  -> train.json      full history, rewritten every epoch; epochs_completed==50 = done-sentinel')

## Cell 3 — CIFAR-10 (reuse the uploaded tarball; hash-checked)

In [ ]:
import hashlib, tarfile, os
os.makedirs(f'{REPO}/data', exist_ok=True)
if not os.path.exists(f'{REPO}/data/cifar-10-batches-py/train_batch') and \
   not os.path.exists(f'{REPO}/data/cifar-10-batches-py/data_batch_1'):
    if os.path.exists(CIFAR_TARGZ):
        h = hashlib.sha256(open(CIFAR_TARGZ, 'rb').read()).hexdigest()
        assert h.startswith('6d958be074577803'), f'CIFAR tarball sha mismatch: {h[:16]}'
        with tarfile.open(CIFAR_TARGZ) as t: t.extractall(f'{REPO}/data')
        print('extracted the uploaded CIFAR-10 tarball (hash verified)')
    else:
        import torchvision
        torchvision.datasets.CIFAR10(f'{REPO}/data', train=True, download=True)
        torchvision.datasets.CIFAR10(f'{REPO}/data', train=False, download=True)
        print('downloaded CIFAR-10 (no tarball on Drive)')
assert os.path.exists(f'{REPO}/data/cifar-10-batches-py/data_batch_1'), 'CIFAR-10 missing'
print('CIFAR-10 ready at', f'{REPO}/data')

## Cell 4 — Resume status (read-only; just tells you where you are)

In [ ]:
import json, os, torch
latest = f'{OUTDIR}/ckpt_latest.pt'
tj     = f'{OUTDIR}/train.json'
if os.path.exists(latest):
    st = torch.load(latest, map_location='cpu', weights_only=False)
    done = int(st['epoch']) + 1
    print(f'RESUME: {done}/50 epochs already done (best valWU so far {float(st["best"]):.4f}).')
    print(f'        the training cell will continue from epoch {done} -- nothing to do.')
elif os.path.exists(tj):
    print('train.json exists but ckpt_latest.pt does not -- fresh start (history will be overwritten).')
else:
    print('FRESH START: no ckpt_latest.pt on Drive; training begins at epoch 0.')

## Cell 5 — Train (streams live; prints pace + ETA after 2 epochs)

Safe to interrupt or lose the runtime at any point: re-run the notebook and it resumes from
`ckpt_latest.pt` on Drive. Re-running when already finished is a no-op.

In [ ]:
import subprocess, re, os, sys

cmd = [sys.executable, f'{REPO}/scripts/dev/train_full_msd.py',
       '--outdir', OUTDIR, '--seed', str(SEED), '--wandb-mode', 'online']
env = dict(os.environ, ATTACKDRO_ROOT=REPO, C5_NUM_WORKERS='2')
print('$', ' '.join(cmd), '\n')

EP = re.compile(r'ep(\d+)/(\d+).*?\((\d+)s\)')
secs, warned = [], False
proc = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
    m = EP.search(line)
    if m:
        done, total, sec = int(m.group(1)), int(m.group(2)), int(m.group(3))
        secs.append(sec)
        if len(secs) == 2 or (len(secs) > 2 and done % 10 == 0):
            pace = sum(secs[-2:]) / 2
            rem_h = (total - done) * pace / 3600
            print(f'\n>>> pace {pace:.0f}s/epoch | {total - done} epochs left '
                  f'| ETA {rem_h:.1f}h (finishes ~{rem_h:.1f}h from now)\n')
            if rem_h > 24 and not warned:
                warned = True
                print('>>> WARNING: ETA exceeds 24h. This is FINE -- the run is resume-safe.\n'
                      '>>> Colab will drop the runtime before then; just re-open this notebook\n'
                      '>>> and run all cells again. It continues from the last finished epoch.\n'
                      '>>> Expect to re-attach the session a few times. Consider an A100.\n')
rc = proc.wait()
print(f'\n[exit {rc}]')

## Cell 6 — Done? → hand off to the audit

The done-sentinel is `train.json` reaching `epochs_completed == 50`.

In [ ]:
import json, os
tj = f'{OUTDIR}/train.json'
JOB = [
    "     {'name':'M1a_full',",
    "      'ckpt': f'{DRIVE}/C5_full/M1a_full/val_best.pt',",
    "      'family':'robustdro', 'scales':['10k'], 'enabled':True, 'gate':'done',",
    "      'note':'full-scale MSD base + pull-push term'},",
]
if not os.path.exists(tj):
    print('train.json not on Drive yet -- training has not completed an epoch.')
else:
    d = json.load(open(tj))
    done = d.get('epochs_completed', 0)
    print(f'epochs_completed: {done}/50 | best val worst-union (proxy): {d.get("best_val_worst_union"):.4f}')
    if done < 50:
        print(f'\nNOT finished. Re-run Cell 5 to continue from epoch {done}.')
    else:
        print('\nDONE. The proxy number above is NOT the reported result -- audit next:\n')
        print('  1. Open eval_colab.ipynb (GPU runtime).')
        print('  2. Add this job to its QUEUE:')
        for ln in JOB:
            print(ln)
        print('  3. Result lands at union_bench/M1a_full/10k/{eval.json, masks_multinorm_v1.npz}.')
        print('\n  NOTE: that notebook_s done-gate requires epoch >= 70, which fits the 80-epoch')
        print('  arms. This run is 50 epochs BY DESIGN, so the gate would wrongly skip it.')
        print('  Pass min_epoch=45 for this job, or drop the gate key (train.json already')
        print('  proves completion). Do not raise the epoch count to satisfy the gate.')

---
## H3 ABLATION — single-view glue (pull x_MSD) · `M1a_msdglue`

**Separate run from M1a-FULL above.** Run the setup cells (GPU · Drive · code · W&B · CIFAR)
but **SKIP Cell 5** (the 50-ep M1a-FULL train) — this is an independent **80-ep from-scratch**
arm on the `c5_fromscratch` recipe, the *same* recipe that produced M1a / M0.

**What it tests (defend §3.3 novelty, H3):** is CLAMP's 3-view multi-norm glue *necessary*, or
does pulling the single **union adversarial** `x_MSD` suffice? `M1a_msdglue` is identical to M1a
in every hyper-parameter — 80 ep · lr 0.05→0.005@70 · bs 128 · α=β=0.5 · τ=0.1 · warmup 10 ·
head 512→128 (eval-discarded) · val-select worst-union parity · seed 0 · triple 8/255·0.5·12 —
**except the pull-push view source**: one view = `x_MSD` (the CE-AT base union adversarial,
*reused*) instead of 3 per-norm APGD views. No fork — `c5_fromscratch.py --glue-view msd`.

Base CE stays MSD `msd_v0` 10-step (unchanged). Glue pulls `h(f(x_MSD)) →` stop-grad
`h(f(x_clean))`; scaffold pushes `h(f(x_MSD))` off other-class `h(f(x_MSD))` (neg = adv).
Dropping the 3 extra APGD attacks makes it **~4× cheaper on attacks than M1a**. Pre-registered:
**report regardless** (paper-relevant ablation, §5.3 — not pure explore).

**Reading (director):** `paired('M1a_msdglue','M0')` = does the single-view term beat the matched
control? · `paired('M1a','M1a_msdglue')` = what do the 3 views add over one view (**this is H3**)?
Report **union + per-norm**, watching **ℓ₁** (does single-view keep M1a's ℓ₁-heavy signature?).

In [ ]:
# --- H3 ablation config (independent of the M1a_full cells above) ---
# Reuses REPO, DRIVE from the setup cell and the CIFAR data cell. Does NOT touch OUTDIR / M1a_full.
OUTDIR_ABL = f'{DRIVE}/C5_fromscratch/M1a_msdglue'   # val_best.pt (under ckpt/), resume.pt, train.json land HERE
EPOCHS_ABL = 80
SEED_ABL   = 0
import os, torch
os.makedirs(OUTDIR_ABL, exist_ok=True)
assert os.path.exists(f'{REPO}/data/cifar-10-batches-py/data_batch_1'), \
    'CIFAR-10 missing -- run the CIFAR cell (Cell 3) first.'
# c5_fromscratch writes resume.pt every epoch; train.json ONLY at the very end = done-sentinel.
rp = f'{OUTDIR_ABL}/resume.pt'; tj = f'{OUTDIR_ABL}/train.json'
if os.path.exists(tj):
    print('DONE already -- train.json present. Skip the train cell; go to the handoff cell.')
elif os.path.exists(rp):
    st = torch.load(rp, map_location='cpu', weights_only=False)
    print(f'RESUME: last finished epoch {st["epoch"]} (best valWU {float(st["best"]):.4f}) -- train cell continues.')
else:
    print('FRESH START: no resume.pt on Drive; training begins at epoch 0.')
print('OUTDIR_ABL (Drive-direct):', OUTDIR_ABL)

In [ ]:
import subprocess, re, os, sys
# H3 ablation: M1a with single-view glue (x_MSD reused). Identical recipe to M1a otherwise.
cmd = [sys.executable, f'{REPO}/scripts/dev/c5_fromscratch.py',
       '--variant', 'M1a', '--glue-view', 'msd', '--base', 'msd',
       '--seed', str(SEED_ABL), '--epochs', str(EPOCHS_ABL),
       '--outdir', OUTDIR_ABL, '--wandb-mode', 'online']
env = dict(os.environ, ATTACKDRO_ROOT=REPO, C5_NUM_WORKERS='2')
print('$', ' '.join(cmd), '\n')

EP = re.compile(r'ep(\d+)/(\d+).*?\((\d+)s\)')
secs, warned = [], False
proc = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
    m = EP.search(line)
    if m:
        done, total, sec = int(m.group(1)), int(m.group(2)), int(m.group(3))
        secs.append(sec)
        if len(secs) == 2 or (len(secs) > 2 and done % 10 == 0):
            pace = sum(secs[-2:]) / 2
            rem_h = (total - done) * pace / 3600
            print(f'\n>>> pace {pace:.0f}s/epoch | {total - done} epochs left | ETA {rem_h:.1f}h\n')
            if rem_h > 24 and not warned:
                warned = True
                print('>>> ETA > 24h is FINE -- resume-safe. Re-open the notebook and re-run;\n'
                      '>>> it continues from resume.pt (last finished epoch). Consider an A100.\n')
rc = proc.wait()
print(f'\n[exit {rc}]')

In [ ]:
import json, os
tj = f'{OUTDIR_ABL}/train.json'
JOB = [
    "     {'name':'M1a_msdglue',",
    "      'ckpt': f'{DRIVE}/C5_fromscratch/M1a_msdglue/ckpt/val_best.pt',",
    "      'family':'robustdro', 'scales':['1k','10k'], 'enabled':True,",
    "      'note':'H3 single-view glue (x_MSD); pairs vs M0 and vs M1a'},",
]
if not os.path.exists(tj):
    print('train.json not on Drive yet -- the 80-epoch run has not finished. Re-run the train cell.')
else:
    d = json.load(open(tj))
    print(f"DONE -- glue_view={d.get('glue_view')} | best val worst-union (proxy): {d.get('best_val_worst_union'):.4f}")
    print('  (proxy is NOT the reported number -- audit under the frozen 12-AA next.)\n')
    print('  1. Open eval_colab.ipynb (GPU). Add to its QUEUE (fast tier reads @1k first, then @10k):')
    for ln in JOB:
        print(ln)
    print('  2. Masks/eval land at union_bench/M1a_msdglue/{1k,10k}/.')
    print('  3. In the AUTO-PAIRED cell, add the two H3 reads (M1a & M0 masks already present):')
    print("       paired('M1a_msdglue','M0')      # single-view term vs matched control")
    print("       paired('M1a','M1a_msdglue')     # 3-view minus single-view  = H3")
    print('  4. Report union + per-norm from each eval.json (full_audit_per_norm) -- watch the L1 component.')

---
## EFFICIENT CLAMP — dynamic worst-norm glue · `M1a_dynworst`

**Separate 80-ep run** (same setup cells; **SKIP Cell 5** = M1a-FULL). Track-B extension that keeps
CLAMP identity (per-sample, clean anchor). Reuses `c5_fromscratch.py --glue-view dynworst --neg clean`.

**Mechanism:** base CE stays MSD 10-step (→ `x_MSD`). At the end of each epoch, val measures per-norm
robust acc; the **worst norm = argmin acc**. The *next* epoch the glue lane crafts **one** APGD view
of that worst norm (1 attack, not 3) and pulls it → the sample's own **clean** anchor (stop-grad).
Epoch 0 (no val yet) defaults to ℓ∞. Scaffold pushes off other-class **clean** embeddings (`--neg
clean`, = M1b's negatives). α=β=0.5 · τ=0.1 · warmup 10 · head 512→128 (eval-discarded). Everything
else identical to M1a (80 ep · lr 0.05→0.005@70 · bs 128 · SGD 0.9/5e-4 · val-select · seed 0 ·
triple 8/255·0.5·12). **~2 attacks/step vs M1a's 4** (MSD base + 1 dynamic view).

**W&B (per epoch):** `train/{loss,ce,glue,scaffold}` · `val/{acc_clean,acc_linf,acc_l2,acc_l1,union,
worst_union,align,uniformity}` · **`glue/selected_norm`** (0=ℓ∞ · 1=ℓ₂ · 2=ℓ₁ — watch it hop between
norms). Norm names are also in `train.json` history.

**Reading (director):** `paired('M1a_dynworst','M0')` = efficient variant beats control? ·
`paired('M1b','M1a_dynworst')` = **isolate single-view effect** (both neg=clean; only 1-dynamic-view
vs 3-view differs) · `paired('M1a','M1a_dynworst')` = 1 view vs 3. Report union + **per-norm (esp
ℓ₁)** — prediction: some ℓ₁ signature may erode (ℓ₁ aligned only on epochs where it is worst).
Pre-registered: **report regardless**.

> **GPU priority (director):** FT-∞ 10k > CIFAR-100 > this. Run only on a spare/idle GPU.

In [ ]:
# --- Efficient CLAMP (dynworst) config -- independent of the M1a_full cells above ---
OUTDIR_DW = f'{DRIVE}/C5_fromscratch/M1a_dynworst'   # val_best.pt (under ckpt/), resume.pt, train.json land HERE
EPOCHS_DW = 80
SEED_DW   = 0
import os, torch
os.makedirs(OUTDIR_DW, exist_ok=True)
assert os.path.exists(f'{REPO}/data/cifar-10-batches-py/data_batch_1'), \
    'CIFAR-10 missing -- run the CIFAR cell (Cell 3) first.'
rp = f'{OUTDIR_DW}/resume.pt'; tj = f'{OUTDIR_DW}/train.json'
if os.path.exists(tj):
    print('DONE already -- train.json present. Skip the train cell; go to the handoff cell.')
elif os.path.exists(rp):
    st = torch.load(rp, map_location='cpu', weights_only=False)
    print(f'RESUME: last finished epoch {st["epoch"]} (best valU {float(st["best"]):.4f}) -- train cell continues.')
else:
    print('FRESH START: no resume.pt on Drive; training begins at epoch 0 (glue view defaults to Linf).')
print('OUTDIR_DW (Drive-direct):', OUTDIR_DW)

In [ ]:
import subprocess, re, os, sys
# Efficient CLAMP: M1a recipe, glue = 1 dynamic worst-norm view, scaffold neg = clean.
cmd = [sys.executable, f'{REPO}/scripts/dev/c5_fromscratch.py',
       '--variant', 'M1a', '--glue-view', 'dynworst', '--neg', 'clean', '--base', 'msd',
       '--seed', str(SEED_DW), '--epochs', str(EPOCHS_DW),
       '--outdir', OUTDIR_DW, '--wandb-mode', 'online']
env = dict(os.environ, ATTACKDRO_ROOT=REPO, C5_NUM_WORKERS='2')
print('$', ' '.join(cmd), '\n')

EP = re.compile(r'ep(\d+)/(\d+).*?\((\d+)s\)')
secs, warned = [], False
proc = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
    m = EP.search(line)
    if m:
        done, total, sec = int(m.group(1)), int(m.group(2)), int(m.group(3))
        secs.append(sec)
        if len(secs) == 2 or (len(secs) > 2 and done % 10 == 0):
            pace = sum(secs[-2:]) / 2
            rem_h = (total - done) * pace / 3600
            print(f'\n>>> pace {pace:.0f}s/epoch | {total - done} epochs left | ETA {rem_h:.1f}h\n')
            if rem_h > 24 and not warned:
                warned = True
                print('>>> ETA > 24h is FINE -- resume-safe. Re-open and re-run; continues from resume.pt.\n')
rc = proc.wait()
print(f'\n[exit {rc}]')

In [ ]:
import json, os
from collections import Counter
tj = f'{OUTDIR_DW}/train.json'
JOB = [
    "     {'name':'M1a_dynworst',",
    "      'ckpt': f'{DRIVE}/C5_fromscratch/M1a_dynworst/ckpt/val_best.pt',",
    "      'family':'robustdro', 'scales':['1k','10k'], 'enabled':True,",
    "      'note':'efficient CLAMP (dynamic worst-norm glue, neg=clean)'},",
]
if not os.path.exists(tj):
    print('train.json not on Drive yet -- the 80-epoch run has not finished. Re-run the train cell.')
else:
    d = json.load(open(tj))
    norms = [h.get('selected_norm') for h in d['history'] if 'selected_norm' in h]
    print(f"DONE -- glue_view={d.get('glue_view')} neg={d.get('neg_source')} | best val union: {d.get('best_val_worst_union'):.4f}")
    print(f"  selected-norm trajectory ({len(norms)} ep): {dict(Counter(norms))}")
    print('  (proxy is NOT the reported number -- audit under the frozen 12-AA next.)\n')
    print('  1. Open eval_colab.ipynb (GPU). Add to its QUEUE (fast tier @1k first, then @10k):')
    for ln in JOB:
        print(ln)
    print('  2. Masks/eval land at union_bench/M1a_dynworst/{1k,10k}/.')
    print('  3. In the AUTO-PAIRED cell add three reads (M1a/M0/M1b masks already present):')
    print("       paired('M1a_dynworst','M0')     # efficient variant vs matched control")
    print("       paired('M1b','M1a_dynworst')    # isolate single-view effect (both neg=clean)")
    print("       paired('M1a','M1a_dynworst')    # 1 dynamic view vs 3 views")
    print('  4. Report union + per-norm from each eval.json (full_audit_per_norm) -- watch the L1 component.')

---
## GRADIENT SURGERY — asymmetric PCGrad at the encoder · `M1a_dynworst_gs`

**Separate 80-ep run** (same setup cells; **SKIP Cell 5**). Reuses `c5_fromscratch.py … --grad-surgery`.
The flag is **generic** (any pull-push arm); this cell co-launches it on `dynworst` to fill a spare GPU.
Note: the **−2.2 clean cost** was measured on **canonical M1a** (3-view, neg=adv), so the *cleanest*
surgery test is `--grad-surgery` on plain M1a — see the one-liner at the bottom.

**Mechanism:** two backward passes give the task gradient `g_task = ∂L_CE/∂θ_f` and the rep gradient
`g_clamp = ∂L_rep/∂θ_f` at the shared encoder θ_f. Where they **conflict** (dot<0), `g_clamp` is
projected off `g_task` (Gram–Schmidt); **`g_task` is never touched** → the task is protected. The
classifier θ_cls gets task-grad only, the CLAMP head θ_h gets rep-grad only. Goal: **recover the clean
−2.2 while keeping union.** ~1.5–2× slower/step (2 backward) → ~140s/ep, ETA ~3h.

**W&B (per epoch):** adds `surgery/conflict_frac` (fraction of θ_f tensors that conflict) ·
`surgery/cos_task_clamp` (mean cos(g_task, g_clamp), pre-projection) · `surgery/gclamp_norm_ratio`
(‖projected‖/‖original‖ — how much got cut). Tells you whether the conflict is broad or a few tensors.

**Reading (director, pre-registered):** `paired('M1a_dynworst_gs','M1a_dynworst')` — primary is the
**clean Δ** (want ≥0, recovered) and **union Δ** (want ≥0, not lost). **Honest risk / Open Q in the
method (not a bug):** if union *drops* as clean recovers, the conflict **is** the mechanism — the term
buys union by sacrificing clean, and surgery self-refutes. That is still a clean finding — **report
either direction.**

> **Cleanest isolation test (canonical M1a):** in the train cell set `--glue-view 3norm --neg adv` and
> `OUTDIR_GS = …/M1a_gs`, then `paired('M1a_gs','M1a')`.
> **GPU priority (director):** FT-∞ 10k > CIFAR-100 > these — spare GPU only.

In [ ]:
# --- Gradient-surgery run config -- independent of the cells above ---
OUTDIR_GS = f'{DRIVE}/C5_fromscratch/M1a_dynworst_gs'   # val_best.pt (under ckpt/), resume.pt, train.json HERE
EPOCHS_GS = 80
SEED_GS   = 0
import os, torch
os.makedirs(OUTDIR_GS, exist_ok=True)
assert os.path.exists(f'{REPO}/data/cifar-10-batches-py/data_batch_1'), \
    'CIFAR-10 missing -- run the CIFAR cell (Cell 3) first.'
rp = f'{OUTDIR_GS}/resume.pt'; tj = f'{OUTDIR_GS}/train.json'
if os.path.exists(tj):
    print('DONE already -- train.json present. Skip the train cell; go to the handoff cell.')
elif os.path.exists(rp):
    st = torch.load(rp, map_location='cpu', weights_only=False)
    print(f'RESUME: last finished epoch {st["epoch"]} (best valU {float(st["best"]):.4f}) -- train cell continues.')
else:
    print('FRESH START: no resume.pt; training begins at epoch 0 (surgery active once warmup ramps the term).')
print('OUTDIR_GS (Drive-direct):', OUTDIR_GS)

In [ ]:
import subprocess, re, os, sys
# Gradient surgery (asymmetric PCGrad) co-launched on the dynworst arm. --grad-surgery gates the
# 2-backward path (task-grad protected). Flag is generic -- see the markdown for the M1a_gs test.
cmd = [sys.executable, f'{REPO}/scripts/dev/c5_fromscratch.py',
       '--variant', 'M1a', '--glue-view', 'dynworst', '--neg', 'clean', '--base', 'msd',
       '--grad-surgery',
       '--seed', str(SEED_GS), '--epochs', str(EPOCHS_GS),
       '--outdir', OUTDIR_GS, '--wandb-mode', 'online']
env = dict(os.environ, ATTACKDRO_ROOT=REPO, C5_NUM_WORKERS='2')
print('$', ' '.join(cmd), '\n>>> ~1.5-2x slower/step (2 backward) -- expect ~140s/ep, ETA ~3h.\n')

EP = re.compile(r'ep(\d+)/(\d+).*?\((\d+)s\)')
secs, warned = [], False
proc = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
    m = EP.search(line)
    if m:
        done, total, sec = int(m.group(1)), int(m.group(2)), int(m.group(3))
        secs.append(sec)
        if len(secs) == 2 or (len(secs) > 2 and done % 10 == 0):
            pace = sum(secs[-2:]) / 2
            rem_h = (total - done) * pace / 3600
            print(f'\n>>> pace {pace:.0f}s/epoch | {total - done} epochs left | ETA {rem_h:.1f}h\n')
            if rem_h > 24 and not warned:
                warned = True
                print('>>> ETA > 24h is FINE -- resume-safe. Re-open and re-run; continues from resume.pt.\n')
rc = proc.wait()
print(f'\n[exit {rc}]')

In [ ]:
import json, os
tj = f'{OUTDIR_GS}/train.json'
JOB = [
    "     {'name':'M1a_dynworst_gs',",
    "      'ckpt': f'{DRIVE}/C5_fromscratch/M1a_dynworst_gs/ckpt/val_best.pt',",
    "      'family':'robustdro', 'scales':['1k','10k'], 'enabled':True,",
    "      'note':'gradient surgery (asymmetric PCGrad) on dynworst'},",
]
if not os.path.exists(tj):
    print('train.json not on Drive yet -- the 80-epoch run has not finished. Re-run the train cell.')
else:
    d = json.load(open(tj))
    cf = [h.get('surg_conflict_frac') for h in d['history'] if 'surg_conflict_frac' in h]
    print(f"DONE -- grad_surgery={d.get('grad_surgery')} | best val union: {d.get('best_val_worst_union'):.4f}")
    if cf:
        print(f"  surgery/conflict_frac: first={cf[0]:.3f}  last={cf[-1]:.3f}  mean={sum(cf)/len(cf):.3f}")
    print('  (proxy is NOT the reported number -- audit under the frozen 12-AA next.)\n')
    print('  1. Open eval_colab.ipynb (GPU). Add to its QUEUE (fast tier @1k first, then @10k):')
    for ln in JOB:
        print(ln)
    print('  2. Masks/eval land at union_bench/M1a_dynworst_gs/{1k,10k}/.')
    print('  3. AUTO-PAIRED cell -- primary readout (clean Delta want >=0, union Delta want >=0):')
    print("       paired('M1a_dynworst_gs','M1a_dynworst')")
    print('  4. Report clean + union + per-norm. Pre-registered: report EITHER direction -- if union')
    print('     drops as clean recovers, the conflict IS the mechanism (term buys union by')
    print('     sacrificing clean, surgery self-refutes). Still a clean finding.')

---
## PHASED PULL–PUSH — temporal decomposition · `M1a_pp_push2pull`

**Separate 80-ep run** (same setup cells; **SKIP Cell 5**). Reuses `c5_fromscratch.py --pp-schedule
push_then_pull --pp-switch 40`. Protocol variant — **loss terms unchanged**, only their timing.

**Mechanism:** ep 1–40 **push only** (L = CE + α·scaffold, β=0, glue off) → ep 41–80 **pull only**
(L = CE + β·glue, α=0, scaffold off). **CE on x_MSD stays on all 80 ep** (backbone keeps classes
separated → anti-collapse). The phase-1 term (push) warms up 0→0.5 over the first 10 ep; **phase 2
(glue) enters at full β=0.5 with no re-warmup** (encoder already warm). The push-only phase reuses the
single x_MSD view (no 3-APGD) → **phase 1 is markedly cheaper than M1a**; the 3 APGD norm-views are
crafted only in the pull phase. neg=adv, τ=0.1, head 512→128 shared by both terms; everything else = M1a.

**Pre-registered deviation (report regardless):** the push-only phase scaffolds on **`z_MSD`** — own
and other-class MSD-adv embeddings (project `x_MSD`'s CE-lane feature through the head) — instead of the
3-view APGD negatives that simul M1a uses; **no APGD view is crafted in the push phase**. Justification:
**M1b** (scaffold is robust to the negative source) + **H3** (`z_MSD` is a valid single view). This makes
phase 1 strictly a single-view push. Recorded in `train.json` (`pp_schedule_note`).

**W&B (per epoch):** `train/{loss,ce,glue,scaffold}` · `val/{acc_clean,acc_linf,acc_l2,acc_l1,union,
worst_union,align,uniformity}` · **`sched/active_term`** (0=push, 1=pull). **Watch:** `val/align`
around the switch (ep 40–45) — when glue turns on, align should drop fast — and whether `val/union`
**dips** at the switch.

**Reading (director, pre-registered):** `paired('M1a_pp_push2pull','M0')` = does phased add over the
control? · `paired('M1a','M1a_pp_push2pull')` = **simultaneous vs phased**, primary readout **union Δ
AND clean Δ** (does phasing recover the −2.2 clean of simul?). If phased ≥ simul union OR recovers
clean → within-step interference is real (temporal analogue of gradient surgery). If simul wins → the
two terms are synergistic when run together. **Report either direction.** (Director's prior: simul
wins union slightly, phased recovers part of the clean.)

> **GPU priority:** after dynworst; any spare Colab GPU. **Do NOT touch the 5070ti** (reserved for
> per-class masks + APGD-ℓ∞@10k triage).

In [ ]:
# --- Phased pull-push config -- independent of the cells above ---
OUTDIR_PP = f'{DRIVE}/C5_fromscratch/M1a_pp_push2pull'   # val_best.pt (under ckpt/), resume.pt, train.json HERE
EPOCHS_PP = 80
SWITCH_PP = 40
SEED_PP   = 0
import os, torch
os.makedirs(OUTDIR_PP, exist_ok=True)
assert os.path.exists(f'{REPO}/data/cifar-10-batches-py/data_batch_1'), \
    'CIFAR-10 missing -- run the CIFAR cell (Cell 3) first.'
rp = f'{OUTDIR_PP}/resume.pt'; tj = f'{OUTDIR_PP}/train.json'
if os.path.exists(tj):
    print('DONE already -- train.json present. Skip the train cell; go to the handoff cell.')
elif os.path.exists(rp):
    st = torch.load(rp, map_location='cpu', weights_only=False)
    ph = 'push' if st['epoch']+1 < SWITCH_PP else 'pull'
    print(f'RESUME: last finished epoch {st["epoch"]} (phase now: {ph}, best valU {float(st["best"]):.4f}).')
else:
    print(f'FRESH START: no resume.pt; ep 1-{SWITCH_PP} push then ep {SWITCH_PP+1}-{EPOCHS_PP} pull.')
print('OUTDIR_PP (Drive-direct):', OUTDIR_PP)

In [ ]:
import subprocess, re, os, sys
# Phased pull-push: ep1-switch push-only, ep switch+1-80 pull-only; CE on x_MSD all 80 ep.
cmd = [sys.executable, f'{REPO}/scripts/dev/c5_fromscratch.py',
       '--variant', 'M1a', '--pp-schedule', 'push_then_pull', '--pp-switch', str(SWITCH_PP),
       '--neg', 'adv', '--base', 'msd',
       '--seed', str(SEED_PP), '--epochs', str(EPOCHS_PP),
       '--outdir', OUTDIR_PP, '--wandb-mode', 'online']
env = dict(os.environ, ATTACKDRO_ROOT=REPO, C5_NUM_WORKERS='2')
print('$', ' '.join(cmd), '\n>>> phase 1 (push) is cheaper -- single x_MSD view; phase 2 (pull) crafts 3 APGD views.\n')

EP = re.compile(r'ep(\d+)/(\d+).*?\((\d+)s\)')
secs, warned = [], False
proc = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
    m = EP.search(line)
    if m:
        done, total, sec = int(m.group(1)), int(m.group(2)), int(m.group(3))
        secs.append(sec)
        if len(secs) == 2 or (len(secs) > 2 and done % 10 == 0):
            pace = sum(secs[-2:]) / 2
            rem_h = (total - done) * pace / 3600
            print(f'\n>>> pace {pace:.0f}s/epoch | {total - done} epochs left | ETA {rem_h:.1f}h\n')
            if rem_h > 24 and not warned:
                warned = True
                print('>>> ETA > 24h is FINE -- resume-safe. Re-open and re-run; continues from resume.pt.\n')
rc = proc.wait()
print(f'\n[exit {rc}]')

In [ ]:
import json, os
tj = f'{OUTDIR_PP}/train.json'
JOB = [
    "     {'name':'M1a_pp_push2pull',",
    "      'ckpt': f'{DRIVE}/C5_fromscratch/M1a_pp_push2pull/ckpt/val_best.pt',",
    "      'family':'robustdro', 'scales':['1k','10k'], 'enabled':True,",
    "      'note':'phased pull-push (push ep1-40 then pull ep41-80)'},",
]
if not os.path.exists(tj):
    print('train.json not on Drive yet -- the 80-epoch run has not finished. Re-run the train cell.')
else:
    d = json.load(open(tj)); hs = d['history']; sw = d['pp_switch']
    near = [(h['epoch']+1, round(h.get('val_align', float('nan')), 3), round(h.get('val_union', float('nan')), 3))
            for h in hs if sw-2 <= h['epoch'] <= sw+3]
    print(f"DONE -- pp={d.get('pp_schedule')}@{sw} | best val union: {d.get('best_val_worst_union'):.4f}")
    print(f"  (ep, align, union) around the switch: {near}")
    print('  (proxy is NOT the reported number -- audit under the frozen 12-AA next.)\n')
    print('  1. Open eval_colab.ipynb (GPU). Add to its QUEUE (fast tier @1k first, then @10k):')
    for ln in JOB:
        print(ln)
    print('  2. Masks/eval land at union_bench/M1a_pp_push2pull/{1k,10k}/.')
    print('  3. AUTO-PAIRED cell -- primary readout is BOTH union Delta and clean Delta:')
    print("       paired('M1a_pp_push2pull','M0')       # phased vs matched control")
    print("       paired('M1a','M1a_pp_push2pull')      # simul vs phased (does phasing recover clean?)")
    print('  4. Report union + clean + per-norm. Pre-registered: report either direction.')

---
## BASE-GENERALITY + REDUNDANCY ablations · #11–#14

**Separate 80-ep from-scratch runs** (same setup cells; **SKIP Cell 5** = M1a-FULL). Same c5_fromscratch
recipe as M1a/M0, matched EXACTLY except `--base`/`--glue-view`/`--variant`. Zip **sha 6a991d47+**.

- **#11** `M0_max`/`M1a_max` (base=MAX worst-of-3, 3 adv/step) · **#12** `M0_avg`/`M1a_avg` (base=MEAN-of-3)
- **#13** `M1a_max_msdglue` (base=MAX + single MSD glue, 4 adv/step) · **#14** `M1a_linfglue` (base=MSD + single ℓ∞ glue, 2 adv/step)

Loop runs all 6 sequentially, **resume-safe** (skip-if-`train.json`; re-run continues). ~8 h/arm — run in
several sessions (each arm resumes). Eval + paired in `eval_colab` (no-Sq@1k → paired + per-norm ℓ₁).

In [ ]:
import subprocess, os, sys
ABL = {
  'M0_max':          ['--variant','M0','--base','max'],
  'M1a_max':         ['--variant','M1a','--base','max','--glue-view','3norm','--neg','adv'],
  'M0_avg':          ['--variant','M0','--base','avg'],
  'M1a_avg':         ['--variant','M1a','--base','avg','--glue-view','3norm','--neg','adv'],
  'M1a_max_msdglue': ['--variant','M1a','--base','max','--glue-view','msd','--neg','adv'],
  'M1a_linfglue':    ['--variant','M1a','--base','msd','--glue-view','linf','--neg','adv'],
  'M1a_cleance':     ['--variant','M1a','--base','clean','--glue-view','3norm','--neg','adv'],  # CLEAN-CE necessity ablation
}
for name, flags in ABL.items():
    out=f'{DRIVE}/C5_ablations/{name}'; os.makedirs(out, exist_ok=True)
    if os.path.exists(f'{out}/train.json'): print(f'SKIP {name} (done)'); continue
    cmd=[sys.executable,f'{REPO}/scripts/dev/c5_fromscratch.py','--seed','0','--epochs','80',
         '--outdir',out,'--wandb-mode','online']+flags
    print('\n$',' '.join(cmd),'\n')
    env=dict(os.environ,ATTACKDRO_ROOT=REPO,C5_NUM_WORKERS='2')
    p=subprocess.Popen(cmd,cwd=REPO,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout: print(line,end='')
    print('[exit',p.wait(),']')

---
## M1a_cleance — CLEAN-CE + CLAMP (necessity ablation)

Base CE on **clean** images; the CLAMP term is unchanged (3 per-norm APGD-10 views, α=β=0.5, τ=0.1, warmup 10).
The classifier head `g` therefore never sees an adversarial example.

**Pre-registered expectation: union COLLAPSES.** A negative result is the expected result and is reported either way.
Reference marks @1k no_square: undefended ≈ 0 · MSD control (M0) **0.3920** · MSD+CLAMP (M1a) **0.4360**.

Deferred off the local GPU because it is an 80-epoch train that is not on the critical path.

`outdir` lives on **Drive** so `resume.pt` survives a session drop — if Colab disconnects, just re-run the
training cell and it picks up from the last completed epoch. Nothing else to do.


In [ ]:
# ── M1a_cleance : TRAIN (resume-safe, re-run this cell after any session drop) ──
import os, sys, subprocess, json, torch

ARM    = 'M1a_cleance'
OUTDIR = f'{DRIVE}/C5_ablations/{ARM}'        # on Drive -> resume.pt survives a dead session
ARGS   = ['--variant','M1a', '--base','clean', '--glue-view','3norm', '--neg','adv',
          '--seed','0', '--epochs','80']       # everything else = headline defaults

# guard: this cell must not silently become a different arm
assert ARGS[ARGS.index('--variant')+1] == 'M1a' and ARGS[ARGS.index('--base')+1] == 'clean', 'wrong arm'
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch runtime to GPU"}')
print(f'W&B run will be: C5_M1a_advneg_seed0_clean   (distinct from every existing run)')

os.makedirs(OUTDIR, exist_ok=True)
rp = f'{OUTDIR}/resume.pt'
if os.path.exists(rp):
    st = torch.load(rp, map_location='cpu', weights_only=False)
    print(f'RESUMING from epoch {st.get("epoch")}  (best_valWU={st.get("best"):.4f})')
else:
    print('fresh start (no resume.pt yet)')

cmd = [sys.executable, f'{REPO}/scripts/dev/c5_fromscratch.py',
       *ARGS, '--outdir', OUTDIR, '--wandb-mode', 'online']
print('\n$', ' '.join(cmd), '\n', flush=True)
p_ = subprocess.Popen(cmd, cwd=REPO, env=dict(os.environ, ATTACKDRO_ROOT=REPO, C5_NUM_WORKERS='2'),
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p_.stdout: print(line, end='')
rc = p_.wait(); print('[exit', rc, ']')

tj = f'{OUTDIR}/train.json'
if rc == 0 and os.path.exists(tj):
    d = json.load(open(tj))
    print(f'\nDONE  epochs={len(d.get("history") or [])}/{d["epochs"]}  best_val_worst_union={d.get("best_val_worst_union")}')
    print('c5_fromscratch writes train.json only at the end; len(history)==epochs is the completion signal.')
else:
    print('\nnot finished (session drop or error). Re-run this cell — it resumes from resume.pt.')


In [ ]:
# ── M1a_cleance : EVAL @1k tier=no_square (9 attacks) — run only after epochs_completed==80 ──
import os, sys, subprocess, json, shutil

ARM    = 'M1a_cleance'
OUTDIR = f'{DRIVE}/C5_ablations/{ARM}'
CKPT   = f'{OUTDIR}/ckpt/val_best.pt'

d = json.load(open(f'{OUTDIR}/train.json'))
# c5_fromscratch.py writes train.json ONCE, after the loop (scripts/dev/c5_fromscratch.py:~700),
# and there is no `epochs_completed` key — completion is len(history)==epochs.
# (train_full_msd.py is the one that rewrites per epoch and carries epochs_completed; different script.)
n_done = len(d.get('history') or [])
assert n_done == d['epochs'], f'training not finished: {n_done}/{d["epochs"]} epochs in history — re-run the train cell.'
assert os.path.exists(CKPT), f'missing {CKPT}'

cmd = [sys.executable, f'{REPO}/scripts/dev/eval_arm.py', '--arm', ARM, '--ckpt', CKPT,
       '--scale', '1k', '--tier', 'no_square']
print('$', ' '.join(cmd), '\n', flush=True)
p_ = subprocess.Popen(cmd, cwd=REPO, env=dict(os.environ, ATTACKDRO_ROOT=REPO),
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p_.stdout: print(line, end='')
rc = p_.wait(); print('[exit', rc, ']')
assert rc == 0, 'eval failed — do not copy partial results back'

src = f'{REPO}/results/eval/union_bench/{ARM}/1k_nosq'
dst = f'{DRIVE}/audit_out/{ARM}_1k_nosq'
os.makedirs(dst, exist_ok=True)
for f in ('eval.json','masks_multinorm_v1.npz'):
    if os.path.exists(f'{src}/{f}'):
        shutil.copy2(f'{src}/{f}', f'{dst}/{f}'); print(f'  -> {dst}/{f}')

e = json.load(open(f'{src}/eval.json')); pn = e.get('per_norm_audit') or {}
u = e['full_audit_union']
print('\n' + '='*62)
print(f"  arm      : {ARM}  (1k, tier=no_square)")
print(f"  n_attacks: {len(e.get('per_attack',{}))}   (must be 9)")
print(f"  clean    : {e.get('clean_acc'):.4f}")
print(f"  union    : {u:.4f}")
print(f"  per-norm : l_inf {pn.get('audit_acc_linf',float('nan')):.4f}  "
      f"l2 {pn.get('audit_acc_l2',float('nan')):.4f}  l1 {pn.get('audit_acc_l1',float('nan')):.4f}")
print('='*62)
print(f"  vs MSD control  (M0)  0.3920   -> {u-0.3920:+.4f}")
print(f"  vs MSD+CLAMP    (M1a) 0.4360   -> {u-0.4360:+.4f}")
print('\n  Pre-registered expectation was a COLLAPSE. If union > 0.30 the collapse did NOT happen')
print('  and a 12-AA @10k run is warranted; otherwise 1k screening is enough. Report either way.')
print(f'\nSend back: this printout + share links to {dst}/')


<!-- P1 LOCUS — alignment term on the classifier-facing features -->
---
## P1 LOCUS PILOT — CLAMP term on `f` instead of on the projection head · arm `E`

**Pre-registered**: `docs/preregistrations/preregistration_P1_locus.md` (committed 2026-07-23,
before any P1 code existed). **Read §4 before reading any number** — the `E − M1a` contrast is
confounded by construction and its reading rule is asymmetric and fixed in advance.

**Separate 80-ep run** (same setup cells 1–8: GPU · Drive · code · W&B · CIFAR; **SKIP Cell 5**).

**What changes vs `M1a`, and only this:** glue and scaffold are computed on `normalize(f)`, where
`f` is the pooled 512-d **post-ReLU** encoder output the linear classifier consumes, instead of on
`normalize(head(f))`. `scripts/dev/c5_fromscratch.py` is **not edited** — `p1_locus_train.py`
imports it and swaps one function at runtime; with the default `--clamp-space h` it swaps nothing.

**Known property, recorded and NOT tuned around** (pre-reg §7.1): `f ≥ 0` after the final ReLU, so
cosine on `f` is confined to `[0,1]` and the push branch cannot pass orthogonality. No signed
projection, no τ retune, no Euclidean push, no ReLU removal. If it diverges or NaNs → **report and
stop**, do not rescue.

**The head must stay frozen** (pre-reg §7.2). Asserted automatically: `.grad is None` every epoch,
head `state_dict` bitwise identical to init at the end, and `head[2].weight.std() == 0.02549`
(T2b's untrained-head value). The train cell fails loudly if the head moved.

### ⏱ Budget before you commit
`M1a` seed 0 took **365 s/epoch × 80 = 8.11 h** on a local 5070 Ti. A Colab **T4 is ~3–5× slower
→ 24–40 h** (many sessions); **L4/A100** is far closer to the local figure. The train cell prints a
measured per-epoch pace and a full-run ETA **after epoch 1** — decide then. Every epoch writes
`resume.pt` to Drive, so interruption is safe and re-running the cell continues.

### Before you start
Rebuild and re-upload `attackdro_code.zip` so it contains the three new files:
`scripts/dev/p1_locus_train.py`, `scripts/dev/p1_locus_eval.py`, `scripts/dev/make_table1.py`.
The eval cell also needs the frozen 1k masks of `M0` and `M1a` (`results/main/{M0,M1a}/1k/`);
it checks for them and tells you exactly what to upload if they are absent.


In [ ]:
# ── P1 arm E : TRAIN (resume-safe — re-run this cell after any session drop) ──
import os, sys, subprocess, json, time, re, torch

ARM    = 'E'
OUTDIR = f'{DRIVE}/P1_locus/{ARM}'            # resume.pt + ckpt/val_best.pt + train.json land on Drive
EPOCHS = 80
# recipe = the weak-base C5 recipe VERBATIM (pre-registration §7); ONE change: --clamp-space f
ARGS = ['--clamp-space', 'f', '--variant', 'M1a', '--base', 'msd', '--glue-view', '3norm',
        '--neg', 'adv', '--seed', '0', '--epochs', str(EPOCHS)]

TRAINER = f'{REPO}/scripts/dev/p1_locus_train.py'
assert os.path.exists(TRAINER), (
    'p1_locus_train.py is not in the code zip — rebuild and re-upload attackdro_code.zip')
assert ARGS[ARGS.index('--clamp-space') + 1] == 'f', 'this cell must run the f-locus arm'

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE — Runtime > Change runtime type > GPU')
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM: {free/2**30:.1f} GiB free / {total/2**30:.1f} GiB total')

os.makedirs(OUTDIR, exist_ok=True)
rp = f'{OUTDIR}/resume.pt'
done_before = 0
if os.path.exists(rp):
    st = torch.load(rp, map_location='cpu', weights_only=False)
    done_before = int(st.get('epoch', -1)) + 1
    print(f'RESUMING — {done_before}/{EPOCHS} epochs already done, best val worst-union={st.get("best"):.4f}')
else:
    print('fresh start (no resume.pt on Drive yet)')

cmd = [sys.executable, TRAINER, *ARGS, '--outdir', OUTDIR, '--wandb-mode', 'online']
print('\n$', ' '.join(cmd), '\n', flush=True)

t_start, times, pat = time.time(), [], re.compile(r'ep(\d+)/(\d+).*?\((\d+)s\)')
p_ = subprocess.Popen(cmd, cwd=REPO, env=dict(os.environ, ATTACKDRO_ROOT=REPO, C5_NUM_WORKERS='2'),
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p_.stdout:
    print(line, end='')
    m = pat.search(line)
    if m:
        ep, tot, sec = int(m.group(1)), int(m.group(2)), int(m.group(3))
        times.append(sec)
        if len(times) >= 1:                      # ETA from epoch 1, as requested
            pace = sum(times) / len(times)
            left = (tot - ep) * pace
            print(f'    >>> pace {pace:.0f}s/epoch | {tot-ep} epochs left | '
                  f'ETA {left/3600:.1f} h (finish ~{time.strftime("%H:%M", time.localtime(time.time()+left))})',
                  flush=True)
rc = p_.wait()
print('[exit', rc, ']  wall', f'{(time.time()-t_start)/3600:.2f} h')

tj = f'{OUTDIR}/train.json'
if rc == 0 and os.path.exists(tj):
    d = json.load(open(tj)); h = d.get('history') or []
    print(f'\nDONE  epochs={len(h)}/{d["epochs"]}  best_val_worst_union={d.get("best_val_worst_union")}')
    print('(c5_fromscratch writes train.json only at the end; len(history)==epochs is the done-signal)')
    if h:
        print(f'  last epoch f-space diagnostics: cos(view, clean anchor)='
              f'{h[-1].get("val_p1_cos_f_anchor")}   cos(diff-class pair)={h[-1].get("val_p1_cos_f_negpair")}')
else:
    print('\nNOT finished (session drop or error). Re-run this cell — it resumes from resume.pt.')
    print('If it stopped on a NaN/divergence: STOP and report it. Do not retune (pre-registration §7.1).')


In [ ]:
# ── P1 : EVAL — REDUCED harness (APGD-CE ×3) over E, M0, M1a — run after epochs==80 ──
import os, sys, subprocess, json, shutil, importlib.util

ARM, OUTDIR = 'E', f'{DRIVE}/P1_locus/E'
CKPT = f'{OUTDIR}/ckpt/val_best.pt'

d = json.load(open(f'{OUTDIR}/train.json'))
n_done = len(d.get('history') or [])
assert n_done == d['epochs'], f'training not finished: {n_done}/{d["epochs"]} — re-run the train cell'
assert os.path.exists(CKPT), f'missing {CKPT}'

if importlib.util.find_spec('autoattack') is None:            # the audit harness's attack backend
    subprocess.run(['pip', 'install', '-q', 'git+https://github.com/fra31/auto-attack'], check=True)

# the two existing arms are read from their FROZEN 1k masks (read-only, not re-attacked)
need = [f'{REPO}/results/main/{a}/1k/{f}' for a in ('M0', 'M1a')
        for f in ('masks_multinorm_v1.npz', 'eval.json')]
missing = [p for p in need if not os.path.exists(p)]
if missing:
    for p in missing:                       # try Drive before giving up
        src = p.replace(REPO, DRIVE)
        if os.path.exists(src):
            os.makedirs(os.path.dirname(p), exist_ok=True); shutil.copy2(src, p)
    missing = [p for p in need if not os.path.exists(p)]
assert not missing, ('frozen 1k masks for M0/M1a are absent. Upload them to Drive mirroring the repo '
                     'layout, e.g. {DRIVE}/results/main/M0/1k/masks_multinorm_v1.npz. Missing:\n  '
                     + '\n  '.join(missing))

cmd = [sys.executable, f'{REPO}/scripts/dev/p1_locus_eval.py',
       '--e-ckpt', CKPT, '--e-train-json', f'{OUTDIR}/train.json']
print('$', ' '.join(cmd), '\n', flush=True)
p_ = subprocess.Popen(cmd, cwd=REPO, env=dict(os.environ, ATTACKDRO_ROOT=REPO),
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p_.stdout: print(line, end='')
rc = p_.wait(); print('[exit', rc, ']')
assert rc == 0, 'eval failed — do not copy partial results back'


In [ ]:
# ── P1 : final table + copy results back to Drive ──
import os, json, shutil

SRC = f'{REPO}/results/analysis/P1_locus'
DST = f'{DRIVE}/P1_locus/analysis'
os.makedirs(DST, exist_ok=True)
for f in os.listdir(SRC):
    shutil.copy2(f'{SRC}/{f}', f'{DST}/{f}'); print('  ->', f'{DST}/{f}')

r = json.load(open(f'{SRC}/p1_locus.json'))
CELLS = ('union', 'linf', 'l2', 'l1')
print('\n' + '=' * 92)
print('P1 LOCUS PILOT — REDUCED harness: APGD-CE x3, 1k frozen subset, val_best weights')
print('  FAB-T and Square ABSENT -> unions OPTIMISTIC, no black-box gradient-masking check.')
print('  Matches NO arm in the paper. Never table these beside a 12-attack number.')
print('=' * 92)
print(f"  {'arm':<5}{'clean':>9}{'union3':>9}{'linf':>9}{'l2':>9}{'l1':>9}   val_best ep")
for k in ('M0', 'M1a', 'E'):
    if k in r['arms']:
        v = r['arms'][k]
        print(f"  {k:<5}{v['clean_acc']:>9.4f}" + ''.join(f"{v['acc'][c]:>9.4f}" for c in CELLS)
              + f"   {v.get('val_best_epoch') if v.get('val_best_epoch') else '-'}")
print('\n' + '=' * 92)
for key, c in r['contrasts'].items():
    print(f"  {c['label']}")
    for cell in CELLS:
        e = c['cells'][cell]
        lo, hi = e['two_sided_95']
        print(f"    {cell:<7} D={e['delta_plugin']:+.4f}  two-sided95[{lo:+.4f},{hi:+.4f}]  "
              f"lcb95={e['lcb_95']:+.4f}   {'SIGNIFICANT' if e['significant'] else 'ns'}")
    print(f"    clean   D={c['cells']['clean']['delta_plugin']:+.4f}  (unpaired scalar, no CI)")
    print()
print('=' * 92)
print('  READING RULE — pre-registration §4, fixed before any number existed:')
print('    E - M0   : primary, unambiguous.')
print('    E - M1a  : positive AND significant -> evidence for the locus account (E wins despite')
print('               the narrower push range).  ns OR negative -> INCONCLUSIVE: "locus does not')
print('               matter" cannot be separated from "push was constrained". Report as measured.')
print('=' * 92)
print(f'\n  json -> {DST}/p1_locus.json      (send this back)')
